# Retail Sales Forecasting API Walkthrough

This notebook documents the interface layer for the MSML610 project. It imports
utilities from `retail_sales_forecasting_utils.py`, inspects the configuration
dataclasses, and will showcase minimal usage examples for JAX-based recurrent
models.

> Status (Midterm PR): focusing on interface definitions and validation logic.
> Training code and metrics visualizations land in the next milestone.



In [ ]:
from pathlib import Path
from pprint import pprint

from retail_sales_forecasting_utils import (
    DataSourceConfig,
    ModelConfig,
    TemporalFeatureConfig,
    build_feature_pipeline,
    create_rnn_model,
    ensure_data_root,
    prepare_dataloader,
)

DATA_ROOT = Path("/data/store_sales")

data_cfg = DataSourceConfig(
    root_dir=DATA_ROOT,
    sales_file="train.parquet",
    calendar_file="holidays_events.csv",
    oil_file="oil.csv",
    transactions_file="transactions.csv",
    id_columns=("store_nbr", "family"),
    date_column="date",
    target_column="sales",
    frequency="D",
    horizon_days=28,
)

feature_cfg = TemporalFeatureConfig(include_external_regressors=True)
model_cfg = ModelConfig(epochs=5, batch_size=64)

ensure_data_root(data_cfg)

print("DataSourceConfig →")
pprint(data_cfg)
print("\nTemporalFeatureConfig →")
pprint(feature_cfg)
print("\nModelConfig →")
pprint(model_cfg)



## API Building Blocks

The API module surfaces four main pillars used throughout the project:

- **Configuration dataclasses** describing dataset layout, feature toggles, and
  model hyperparameters.
- **Feature pipeline factories** that augment the raw sales data with temporal,
  holiday, promotion, and external regressor signals.
- **Dataloader helpers** that scale numeric columns and materialize sliding
  windows ready for JAX training loops.
- **Training and evaluation utilities** implementing a handcrafted LSTM with
  Optax-based optimization.

The cells below exercise each pillar and surface the artifacts they return.

In [ ]:
pipeline = list(build_feature_pipeline(data_cfg, feature_cfg))
print(f"Pipeline contains {len(pipeline)} steps → {[fn.__name__ for fn in pipeline]}")

train_ds, val_ds, metadata = prepare_dataloader(data_cfg, pipeline, feature_cfg)

print(
    "\nWindow shapes:",
    {
        "train": train_ds.inputs.shape,
        "validation": val_ds.inputs.shape,
        "feature_dim": train_ds.inputs.shape[-1],
        "horizon": train_ds.targets.shape[-1],
    },
)
print("Sample metadata keys:", list(metadata.keys()))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Inspect Prepared Datasets

In [ ]:
train_ds.ids.head()

### Metadata Snapshot


In [ ]:
print("Feature columns (first 10):", metadata["feature_columns"][:10])
print("Target transform:", metadata["target_transform"])
print("Horizon:", metadata["horizon"], "days")
print("Lookback:", metadata["lookback"], "days")


## Instantiate LSTM Model


In [ ]:
model_config = create_rnn_model(model_cfg)
print(model_config)


## Summary

In [ ]:
print("API components initialised and ready for downstream notebooks.")

INFO  > cmd='/venv/lib/python3.12/site-packages/ipykernel_launcher.py -f /home/.local/share/jupyter/runtime/kernel-085a2ce7-6161-4c8a-92d5-492051832f3c.json'


This notebook now hands control to the example workflow, which trains the JAX
model end-to-end and visualises forecasts.

In [ ]:
# Legacy template content removed. See markdown summary above for next steps.